**Bloque 02. Cálculo numérico con NumPy**

Revisión y práctica del *Python Data Science Handbook*, de Jake VanderPlas.
Este cuaderno contiene explicaciones y ejemplos propios en español.

El objetivo es representar mediciones en arreglos y resolver un resumen por estación con operaciones vectorizadas, máscaras y agregaciones.

Requisitos: Python 3, Jupyter y NumPy. Ejecuta las celdas en orden desde un kernel nuevo.
Los datos de los ejemplos son sintéticos, salvo que se indique otra procedencia.
Cada bloque es independiente de los demás.

**1. Datos, tipos y dimensiones**

Un `ndarray` almacena elementos de un tipo definido. Uso una matriz de lluvia
sintética: cada fila es un día y cada columna una estación. `shape` describe
esa organización; `dtype` identifica el tipo de dato. Estas mediciones se
inventaron para el ejercicio y no describen el clima de una localidad.

In [1]:
import numpy as np

print('NumPy:', np.__version__)
estaciones = np.array(['A', 'B', 'C'])
lluvia = np.array([
    [0, 2, 1], [5, 0, 3], [12, 8, 0], [0, 0, 0],
    [3, 1, 4], [18, 13, 9], [2, 0, 1], [0, 4, 2],
    [7, 6, 5], [1, 0, 0], [9, 11, 8], [4, 3, 2],
], dtype=np.float64)
print('Dimensiones:', lluvia.ndim, 'Forma:', lluvia.shape)
print('Elementos:', lluvia.size, 'Tipo:', lluvia.dtype, 'Bytes:', lluvia.nbytes)
print(lluvia)

NumPy: 2.3.5
Dimensiones: 2 Forma: (12, 3)
Elementos: 36 Tipo: float64 Bytes: 288
[[ 0.  2.  1.]
 [ 5.  0.  3.]
 [12.  8.  0.]
 [ 0.  0.  0.]
 [ 3.  1.  4.]
 [18. 13.  9.]
 [ 2.  0.  1.]
 [ 0.  4.  2.]
 [ 7.  6.  5.]
 [ 1.  0.  0.]
 [ 9. 11.  8.]
 [ 4.  3.  2.]]


**2. Creación, selección y reorganización**

`arange`, `linspace` y los constructores de ceros permiten crear arreglos.
Los índices empiezan en cero. Un corte puede compartir memoria con el arreglo
original; para modificar una selección sin alterar su origen uso `.copy()`.

In [2]:
dias = np.arange(1, lluvia.shape[0] + 1)
print('Días:', dias)
print('Cuadrícula:', np.linspace(0, 1, 5))
print('Primeros tres días, estaciones A y B:\n', lluvia[:3, :2])
print('Último día:', lluvia[-1])
print('Días seleccionados:\n', lluvia[[0, 5, 10]])

Días: [ 1  2  3  4  5  6  7  8  9 10 11 12]
Cuadrícula: [0.   0.25 0.5  0.75 1.  ]
Primeros tres días, estaciones A y B:
 [[ 0.  2.]
 [ 5.  0.]
 [12.  8.]]
Último día: [4. 3. 2.]
Días seleccionados:
 [[ 0.  2.  1.]
 [18. 13.  9.]
 [ 9. 11.  8.]]


In [3]:
ejemplo = np.arange(6)
vista = ejemplo[1:4]
copia = ejemplo[1:4].copy()
vista[0] = 99
copia[1] = -10
print('Original después de modificar la vista:', ejemplo)
print('Copia independiente:', copia)
assert ejemplo[1] == 99 and ejemplo[2] == 2

tabla = np.arange(12).reshape(3, 4)
print('Reorganización:\n', tabla)
print('Concatenación:\n', np.concatenate([tabla, tabla], axis=0))
print('Partes:', [parte.shape for parte in np.split(tabla, 2, axis=1)])

Original después de modificar la vista: [ 0 99  2  3  4  5]
Copia independiente: [  1 -10   3]
Reorganización:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
Concatenación:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
Partes: [(3, 2), (3, 2)]


**3. Funciones universales y agregaciones**

Una operación vectorizada aplica el cálculo a todo el arreglo. Con `axis=0`
reduzco los días y obtengo un resultado por estación; con `axis=1` reduzco
las estaciones y obtengo un resultado por día. La suma acumulada conserva
cómo se alcanza el total a lo largo de los días.

In [4]:
lluvia_pulgadas = lluvia / 25.4  # Conversión de milímetros a pulgadas.
total_estacion = lluvia.sum(axis=0)
total_diario = lluvia.sum(axis=1)
print('Totales por estación (mm):', total_estacion)
print('Promedios por estación (mm):', lluvia.mean(axis=0).round(2))
print('Máximo por estación (mm):', lluvia.max(axis=0))
print('Desviación muestral:', lluvia.std(axis=0, ddof=1).round(2))
print('Totales diarios (mm):', total_diario)
print('Acumulado en estación A:', np.cumsum(lluvia[:, 0]))
print('Raíz de los primeros valores:', np.sqrt(lluvia[:2]))
assert np.isclose(total_estacion.sum(), total_diario.sum())

Totales por estación (mm): [61. 48. 35.]
Promedios por estación (mm): [5.08 4.   2.92]
Máximo por estación (mm): [18. 13.  9.]
Desviación muestral: [5.58 4.55 3.06]
Totales diarios (mm): [ 3.  8. 20.  0.  8. 40.  3.  6. 18.  1. 28.  9.]
Acumulado en estación A: [ 0.  5. 17. 17. 20. 38. 40. 40. 47. 48. 57. 61.]
Raíz de los primeros valores: [[0.         1.41421356 1.        ]
 [2.23606798 0.         1.73205081]]


**4. Broadcasting**

La media por columna tiene forma `(3,)` y se resta de cada fila de la matriz
`(12, 3)`. Los tamaños son compatibles desde la última dimensión cuando
coinciden o uno de ellos vale uno. Esta regla permite centrar los datos y
calcular diferencias entre todos los pares de días sin escribir un bucle.

In [5]:
medias = lluvia.mean(axis=0)
desviaciones = lluvia.std(axis=0)
centrada = lluvia - medias
estandarizada = np.divide(
    centrada, desviaciones,
    out=np.zeros_like(centrada), where=desviaciones != 0,
)
print('Media de las columnas centradas:', centrada.mean(axis=0))
print('Primeros días estandarizados:\n', estandarizada[:3].round(2))
assert np.allclose(centrada.mean(axis=0), 0)

diferencias = total_diario[:, None] - total_diario[None, :]
print('Diferencias entre pares, forma:', diferencias.shape)
assert np.allclose(diferencias, -diferencias.T)

Media de las columnas centradas: [3.70074342e-16 0.00000000e+00 2.59052039e-16]
Primeros días estandarizados:
 [[-0.95 -0.46 -0.65]
 [-0.02 -0.92  0.03]
 [ 1.29  0.92 -1.  ]]
Diferencias entre pares, forma: (12, 12)


**5. Comparaciones, lógica y máscaras**

Una comparación produce un arreglo booleano. Para combinar condiciones
elemento a elemento uso `&` y `|`, encerrando cada condición entre paréntesis.
`any` identifica días en que al menos una estación cumple la condición;
`all` exige que la cumplan las tres.

In [6]:
dias_con_lluvia = (lluvia > 0).any(axis=1)
dias_todas = (lluvia > 0).all(axis=1)
evento = (lluvia >= 5) & (lluvia < 12)
print('Días con lluvia en alguna estación:', dias[dias_con_lluvia])
print('Días con lluvia en todas:', dias[dias_todas])
print('Mediciones entre 5 y menos de 12 mm:', lluvia[evento])
print('Eventos por estación:', np.count_nonzero(evento, axis=0))

Días con lluvia en alguna estación: [ 1  2  3  5  6  7  8  9 10 11 12]
Días con lluvia en todas: [ 5  6  9 11 12]
Mediciones entre 5 y menos de 12 mm: [ 5.  8.  9.  7.  6.  5.  9. 11.  8.]
Eventos por estación: [3 3 3]


**6. Índices avanzados y acumulación**

La selección mediante arreglos de índices permite reordenar filas. Cuando
un índice aparece varias veces, `np.add.at` acumula todas las contribuciones;
una asignación aumentada con índices repetidos no expresa lo mismo.

In [7]:
seleccion = lluvia[np.ix_([1, 5, 10], [0, 2])]
print('Días 2, 6 y 11; estaciones A y C:\n', seleccion)

indice_estacion = np.array([0, 1, 0, 2, 1, 0])
conteos = np.zeros(3, dtype=int)
np.add.at(conteos, indice_estacion, 1)
print('Conteo acumulado con índices repetidos:', conteos)
assert np.array_equal(conteos, [3, 2, 1])

Días 2, 6 y 11; estaciones A y C:
 [[ 5.  3.]
 [18.  9.]
 [ 9.  8.]]
Conteo acumulado con índices repetidos: [3 2 1]


**7. Ordenamiento y selección parcial**

`sort` devuelve valores ordenados y `argsort` devuelve sus posiciones.
`partition` separa los valores alrededor de una posición sin ordenar todo
el arreglo. Para mostrar un ranking uso las posiciones de los tres mayores
totales diarios y después recupero sus etiquetas.

In [8]:
orden = np.argsort(total_diario)[::-1]
print('Tres días con mayor total:', dias[orden[:3]])
print('Totales correspondientes:', total_diario[orden[:3]])
print('Tres mayores valores por partición:', np.partition(total_diario, -3)[-3:])

Tres días con mayor total: [ 6 11  3]
Totales correspondientes: [40. 28. 20.]
Tres mayores valores por partición: [20. 28. 40.]


**8. Faltantes, datos estructurados y aleatoriedad**

Un faltante numérico puede representarse con `np.nan`. Las funciones `nanmean`
y similares excluyen esos valores del resumen, por lo que hay que registrar
cuántas observaciones quedan. Un tipo estructurado combina campos de tipos
distintos. Para generar datos reproducibles uso un generador con semilla.

In [9]:
incompleta = lluvia.copy()
incompleta[2, 1] = np.nan
print('Faltantes:', np.isnan(incompleta).sum())
print('Datos disponibles por estación:', np.isfinite(incompleta).sum(axis=0))
print('Media excluyendo faltantes:', np.nanmean(incompleta, axis=0).round(2))

resumen = np.zeros(3, dtype=[('estacion', 'U1'), ('total_mm', 'f8')])
resumen['estacion'] = estaciones
resumen['total_mm'] = total_estacion
print('Arreglo estructurado:', resumen)

rng = np.random.default_rng(42)
muestra_dias = rng.choice(dias, size=4, replace=False)
print('Muestra reproducible de días:', muestra_dias)

Faltantes: 1
Datos disponibles por estación: [12 11 12]
Media excluyendo faltantes: [5.08 3.64 2.92]
Arreglo estructurado: [('A', 61.) ('B', 48.) ('C', 35.)]
Muestra reproducible de días: [ 6  1 11  8]


**9. Ejercicio resuelto y conclusión**

Pregunta: ¿qué estación acumula más lluvia y qué porcentaje de los días
presenta al menos 5 mm en cada estación? El umbral pertenece al ejercicio;
no se interpreta como una alerta meteorológica.

In [10]:
ganadora = estaciones[np.argmax(total_estacion)]
porcentaje = 100 * (lluvia >= 5).mean(axis=0)
print('Estación con mayor acumulado:', ganadora)
for nombre, total, proporcion in zip(estaciones, total_estacion, porcentaje):
    print(f'{nombre}: {total:.1f} mm; {proporcion:.1f}% de los días con >= 5 mm')

assert np.all((porcentaje >= 0) & (porcentaje <= 100))
assert np.allclose(lluvia_pulgadas * 25.4, lluvia)

Estación con mayor acumulado: A
A: 61.0 mm; 41.7% de los días con >= 5 mm
B: 48.0 mm; 33.3% de los días con >= 5 mm
C: 35.0 mm; 25.0% de los días con >= 5 mm


La selección de `axis` define qué pregunta responde el resumen. El broadcasting
expresa operaciones repetidas entre dimensiones compatibles y las máscaras
convierten una condición en una selección de datos. El ejemplo también muestra
por qué distinguir vistas y copias evita modificar datos de forma involuntaria.

**Fuentes y correspondencia con el libro**

Referencia indicada en la consigna: VanderPlas, J. (2022). *Python Data Science Handbook: Essential Tools for Working with Data*. O’Reilly Media.

El material efectivamente revisado es la carpeta `notebooks_v1` del enlace de la consigna. Su prefacio identifica el copyright de 2016; por eso la referencia de 2022 y estos materiales no se presentan como una misma edición. Las secciones siguientes permiten consultar el desarrollo original de los temas practicados aquí.

[02.00-Introduction-to-NumPy](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/02.00-Introduction-to-NumPy.ipynb).

[02.01-Understanding-Data-Types](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/02.01-Understanding-Data-Types.ipynb).

[02.02-The-Basics-Of-NumPy-Arrays](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/02.02-The-Basics-Of-NumPy-Arrays.ipynb).

[02.03-Computation-on-arrays-ufuncs](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/02.03-Computation-on-arrays-ufuncs.ipynb).

[02.04-Computation-on-arrays-aggregates](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/02.04-Computation-on-arrays-aggregates.ipynb).

[02.05-Computation-on-arrays-broadcasting](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/02.05-Computation-on-arrays-broadcasting.ipynb).

[02.06-Boolean-Arrays-and-Masks](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/02.06-Boolean-Arrays-and-Masks.ipynb).

[02.07-Fancy-Indexing](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/02.07-Fancy-Indexing.ipynb).

[02.08-Sorting](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/02.08-Sorting.ipynb).

[02.09-Structured-Data-NumPy](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/02.09-Structured-Data-NumPy.ipynb).

Documentación de las bibliotecas utilizada para adaptar los ejemplos:

[Generadores aleatorios de NumPy](https://numpy.org/doc/stable/reference/random/generator.html).

El texto original del libro declara licencia CC BY-NC-ND y su código, MIT. Las explicaciones y los ejercicios de este cuaderno son una elaboración nueva; las fuentes se conservan como referencia de estudio.